# Tesla (TSLA) Financial Analysis

Interactive walkthrough covering:
- **P&L Analysis** — margins, growth, expense breakdown
- **Budgeting** — forward budget & variance analysis
- **Forecasting** — time-series + scenario modeling
- **Sensitivity Analysis** — tornado charts & data tables

Data source: Yahoo Finance via `yfinance`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.data_fetcher import FinancialDataFetcher
from src.pl_analysis import PLAnalyzer
from src.budgeting import BudgetAnalyzer
from src.forecasting import FinancialForecaster
from src.sensitivity import SensitivityAnalyzer

pd.set_option('display.float_format', lambda x: f'${x/1e9:,.2f}B' if abs(x) > 1e6 else f'{x:,.2f}')

## 1. Data Fetching

In [ ]:
fetcher = FinancialDataFetcher('TSLA')
info = fetcher.get_company_info()
income_stmt = fetcher.get_income_statement()

print(f"Company: {info['name']}")
print(f"Sector: {info['sector']} | Industry: {info['industry']}")
income_stmt[['Total Revenue', 'Gross Profit', 'Operating Income', 'Net Income']]

## 2. P&L Analysis

In [ ]:
pl = PLAnalyzer(income_stmt)
margins = pl.margin_analysis()
growth = pl.yoy_growth()

print('=== Profit Margins ===')
display(margins)
print('\n=== YoY Growth ===')
display(growth)

## 3. Budgeting & Variance

In [ ]:
budget_analyzer = BudgetAnalyzer(income_stmt)
base_year = income_stmt.index[-2]
budget = budget_analyzer.create_budget(
    base_year=base_year,
    growth_assumptions={'Total Revenue': 0.12, 'Operating Expense': 0.08},
    years_forward=2,
)

latest = income_stmt.index[-1]
if latest in budget.index:
    variance = budget_analyzer.variance_analysis(budget, latest)
    print(f'Budget vs Actual ({latest}):')
    display(variance)

## 4. Forecasting

In [ ]:
forecaster = FinancialForecaster(income_stmt)
revenue_fc = forecaster.forecast_metric('Total Revenue', periods=3)
scenarios = forecaster.scenario_forecast('Total Revenue', periods=3)

print('=== Revenue Forecast ===')
display(revenue_fc)
print('\n=== Scenarios ===')
display(scenarios.pivot(index='Year', columns='Scenario', values='Total Revenue'))

## 5. Sensitivity Analysis

In [ ]:
sensitivity = SensitivityAnalyzer(income_stmt)
tornado = sensitivity.tornado_data()
two_way = sensitivity.two_way_sensitivity()

print('=== Tornado Chart Data ===')
display(tornado)
print('\n=== Two-Way Sensitivity (Net Income, $B) ===')
display(two_way / 1e9)